In [3]:
import pandas as pd
import re
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

In [4]:
train_path='/work/data1/shangyifan/cyclic_peptide_cliff/train.csv'
test_path='/work/data1/shangyifan/cyclic_peptide_cliff/test.csv'

In [5]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [6]:
def smiles2fingerprint(smi):
    mol=Chem.MolFromSmiles(smi)
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    fp_array = np.zeros((1,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fingerprint, fp_array)
    return fp_array
                           
train_df['fingerprint']=train_df['SMILES'].apply(smiles2fingerprint)
train_labels = train_df['Permeability'].values
train_df_smi_fg=train_df['fingerprint'].values
train_df_smi_fg = np.vstack(train_df_smi_fg)

test_df['fingerprint']=test_df['SMILES'].apply(smiles2fingerprint)
test_labels =test_df['Permeability'].values
test_df_smi_fg=test_df['fingerprint'].values
test_df_smi_fg = np.vstack(test_df_smi_fg)

In [7]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =train_df_smi_fg,test_df_smi_fg,train_labels,test_labels


# SVR

In [8]:
from sklearn.svm import SVR
regressor = SVR(max_iter=1000)
regressor.fit(X_train, y_train)

# 进行预测
predictions = regressor.predict(X_test)



/home/s2136015/anaconda3/envs/moses/lib/python3.7/site-packages/sklearn/svm/_base.py:289: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  ConvergenceWarning,


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 计算评估指标
mse = mean_squared_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
cur_pearsonr, _ = pearsonr(y_test, predictions)
cur_spearmanr, _ = spearmanr(y_test, predictions)

print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.47159359602721374
Root Mean Squared Error (RMSE): 0.6867267258722451
Mean Absolute Error (MAE): 0.5865358512455177
R^2 Score: 0.16682571181115957
pearsonr: 0.6484351945253121
cur_spearmanr: 0.6318543163365996


# 随机森林

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 创建随机森林回归器
rf_regressor = RandomForestRegressor( random_state=42, n_estimators=5 )

# 使用训练数据拟合模型
rf_regressor.fit(X_train, y_train)

# 使用模型进行预测
y_pred = rf_regressor.predict(X_test)

# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.2391109309393812
Root Mean Squared Error (RMSE): 0.4889897043286098
Mean Absolute Error (MAE): 0.3545613296444217
R^2 Score: 0.5775577078190568
pearsonr: 0.7626237075002326
cur_spearmanr: 0.722714627200738


# MLP

In [11]:
from sklearn.neural_network import MLPRegressor
mlp_regressor = MLPRegressor( random_state=1, max_iter=1000)

# 使用训练数据拟合模型
mlp_regressor.fit(X_train, y_train)

# 使用模型进行预测
y_pred = mlp_regressor.predict(X_test)

# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.2207697428651183
Root Mean Squared Error (RMSE): 0.4698614081461876
Mean Absolute Error (MAE): 0.3441226948041383
R^2 Score: 0.6099614691233766
pearsonr: 0.7906686926055877
cur_spearmanr: 0.7499020106987251


In [12]:
from sklearn.neighbors import KNeighborsRegressor
neigh = KNeighborsRegressor(n_neighbors=2)
neigh.fit(X_train, y_train)
y_pred = neigh.predict(X_test)
# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)


print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.30680288326300986
Root Mean Squared Error (RMSE): 0.5538978996737665
Mean Absolute Error (MAE): 0.3975527426160337
R^2 Score: 0.45796491718646803
pearsonr: 0.7146568231535244
cur_spearmanr: 0.674137017739465


In [13]:
from sklearn.linear_model import ElasticNet
enet = ElasticNet(alpha=0.08, l1_ratio=0.5).fit(X_train, y_train)
enet.fit(X_train, y_train)
y_pred = enet.predict(X_test)
# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)


print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.4737403294258963
Root Mean Squared Error (RMSE): 0.6882879698395842
Mean Absolute Error (MAE): 0.5227566897108027
R^2 Score: 0.16303303293161997
pearsonr: 0.4614214242981334
cur_spearmanr: 0.45130312898945424


In [14]:
from sklearn.tree import DecisionTreeRegressor
regressor = DecisionTreeRegressor(random_state=0)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)
# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)


print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.3384409371376005
Root Mean Squared Error (RMSE): 0.5817567680204507
Mean Absolute Error (MAE): 0.3927731926745586
R^2 Score: 0.40206930444129174
pearsonr: 0.6829183514205406
cur_spearmanr: 0.6531274658839428


In [15]:
from sklearn.ensemble import GradientBoostingRegressor
reg = GradientBoostingRegressor(random_state=0)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)


print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.2969935320336818
Root Mean Squared Error (RMSE): 0.5449711295414481
Mean Absolute Error (MAE): 0.40962678631938154
R^2 Score: 0.475295303555027
pearsonr: 0.6983497451999177
cur_spearmanr: 0.6458713669378668


In [16]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
kernel = DotProduct() + WhiteKernel()
gpr = GaussianProcessRegressor(kernel=kernel,random_state=0)
gpr.fit(X_train, y_train)
y_pred = gpr.predict(X_test)
# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # 设置squared=False来计算RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cur_pearsonr, _ = pearsonr(y_test, y_pred)
cur_spearmanr, _ = spearmanr(y_test, y_pred)


print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("R^2 Score:", r2)
print("pearsonr:", cur_pearsonr)
print("cur_spearmanr:", cur_spearmanr)

Mean Squared Error (MSE): 0.29367373503516403
Root Mean Squared Error (RMSE): 0.5419167233396327
Mean Absolute Error (MAE): 0.3983870515173929
R^2 Score: 0.4811604584775545
pearsonr: 0.6962989986773072
cur_spearmanr: 0.664342523170899
